# Retail Sales Data Analytics: Data Cleaning

## Project Overview and Document Purpose

### Document Purpose

### Executive Overview

## 1. Dataset Preview

Before we start cleaning our dataset, we first need to import the required libraries and load the dataset. This prepares our environment for the data cleaning process.

In [29]:
# modules we'll use
import os
import pandas as pd
import numpy as np

# set project root
project_root = os.getcwd()
if os.path.basename(project_root) == "noteboook":
    project_root = os.path.dirname(project_root)

file_path = os.path.join(project_root, "data", "raw", "retail_sales.xlsx")

try:
    df = pd.read_excel(file_path)
except ImportError as error:
    raise ImportError("Install openpyxl first: pip install openpyxl") from error

# set seed for reproducibility
np.random.seed(0)

Great! Now let’s take a look at our dataset. This helps us confirm that the data was loaded correctly and gives us a better understanding of its structure and contents.

In [30]:
df.shape

(541909, 8)

The dataset contains over `540k rows` and `8 columns`. Next, let's inspect the column names and their data types.

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


It looks like we need to standardize our column names. We should also investigate the `InvoiceNo` and `StockCode` columns, as they are currently stored as an `object` data type. Let's take a closer look at the dataset to better understand these issues.

In [32]:
# preview first 10 rows of the dataset
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


Now we understand why the `StockCode` column is stored as an `object` data type. Although some values appear numeric, the column also contains alphabetic characters, meaning it represents identifiers rather than numerical values. Therefore, converting it to an `int` data type would be inappropriate.

## 2. Standardize Column Names

Before starting the data cleaning process, we need to standardize our column names to maintain consistency throughout the analysis. Currently, some columns such as `InvoiceNo`, `StockCode`, and `Description` use uppercase letters. We will convert all column names to lowercase and replace spaces with underscores (`_`) to follow a consistent naming convention.

In [33]:
# check every columns name
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

In [34]:
# turn column name to lowercase
df.columns = df.columns.str.lower()

# separate column names with underscore ('_')
df = df.rename(columns={
    'invoiceno'     : 'invoice_no',
    'stockcode'     : 'stock_code',
    'invoicedate'   : 'invoice_date',
    'unitprice'     : 'unit_price',
    'customerid'    : 'customer_id'
})

# verify the changes
df.columns

Index(['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date',
       'unit_price', 'customer_id', 'country'],
      dtype='str')

## 3. Data Type Conversion

As mentioned above, the `invoice_no` and `stock_code` columns are both stored as `object` data types. However, `stock_code` contains both numeric and alphabetic characters, so converting it to an integer would result in errors and potentially cause us to lose information.

Therefore, we will keep `stock_code` as it is and convert only `invoice_no` to an integer, since the `invoice_no` column contains only numeric values. However, before making this conversion, we need to verify that all values in `invoice_no` are actually numeric.

In [35]:
# verify invoice_no column
# returns True if every single value is numeric, otherwise False
is_all_numeric = pd.to_numeric(df["invoice_no"], errors="coerce").notna().all()
print(is_all_numeric)

False


In [36]:
# take a look at the column that are not numeric
non_numeric_series = df[pd.to_numeric(df["invoice_no"], errors="coerce").isna()]["invoice_no"]
print(non_numeric_series)

141       C536379
154       C536383
235       C536391
236       C536391
237       C536391
           ...   
540449    C581490
541541    C581499
541715    C581568
541716    C581569
541717    C581569
Name: invoice_no, Length: 9291, dtype: object


After verifying our `invoice_no` column, we can see that it also contains both numeric and alphabetic characters, similar to `stock_code`. Therefore, converting it to an integer would result in errors and could cause us to lose useful information.

## 4. Missing Values

A dataset with missing values can lead to inaccurate results during **Analysis** and **Modeling**. To ensure that our dataset is reliable and ready to use, we need to check for and verify whether it contains any **Null** values.

In [37]:
# check number of missing value
missing_value_count = df.isnull().sum()
missing_value_count

invoice_no           0
stock_code           0
description       1454
quantity             0
invoice_date         0
unit_price           0
customer_id     135080
country              0
dtype: int64

We found missing values in the `customer_id` and `description` columns. Let's determine the total number of missing values in the dataset.

In [39]:
# calculate number of missing values 
total_cells = np.prod(df.shape)
total_missing = missing_value_count.sum()
percent_missing = (total_missing/total_cells) * 100

print("Total values:", df.size)
print("Total missing values:", total_missing)
print("Percentage missing:", percent_missing)

Total values: 4335272
Total missing values: 136534
Percentage missing: 3.149375633178264


The dataset contains over 130K missing values, which account for only 3.14% of the total values. Although this is a relatively small percentage, we should not immediately decide whether to drop or fill the missing values.

First, we need to inspect the missing values to understand their distribution and determine whether the affected columns contain useful information. We should also check whether the missing values occur in the same rows across both columns. This will help us choose the most appropriate strategy for handling them.

In [ ]:
# check what happen when customer_id missing
missing_customer_id = df[df['customer_id'].isnull()]

# view the dataset
missing_customer_id.head(10)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,NaN,United Kingdom
1447,536544,21790,VINTAGE SNAP CARDS,9,2010-12-01 14:32:00,1.66,NaN,United Kingdom
1448,536544,21791,VINTAGE HEADS AND TAILS CARD GAME,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1449,536544,21801,CHRISTMAS TREE DECORATION WITH BELL,10,2010-12-01 14:32:00,0.43,NaN,United Kingdom
1450,536544,21802,CHRISTMAS TREE HEART DECORATION,9,2010-12-01 14:32:00,0.43,NaN,United Kingdom
1451,536544,21803,CHRISTMAS TREE STAR DECORATION,11,2010-12-01 14:32:00,0.43,NaN,United Kingdom


In [ ]:
# check what happen when description missing
missing_description = df[df['description'].isnull()]

# view the dataset
missing_description.head(10)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom


In [ ]:
# check if both missing the same row
missing_both_value = df[df[['customer_id', 'description']].isnull().all(axis=1)]

# view the dataset
missing_both_value.head(10)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom


We can see that both columns have missing values in the same rows. Let's determine how many rows are affected by missing values in both columns.

In [ ]:
# filter for rows where both columns are missing
missing_both = df[df['customer_id'].isnull() & df['description'].isnull()]

# total count of affected rows
affected_rows_count = missing_both.shape[0]

print(f"Number of rows missing both customer_id and description: {affected_rows_count}")


Number of rows missing both customer_id and description: 1454


Now we have a better understanding of the missing values. Both `customer_id` and `description` have the same number of missing values, and these missing values occur in exactly the same rows. In other words, every row where `description` is missing also has a missing `customer_id`.

However, when we look at the other columns, such as `stock_code`, `quantity`, `invoice_date`, and `unit_price`, they still contain useful information. We also noticed some suspicious patterns. For example, some rows have a `unit_price` of `0.0`, while the `invoice_date` and `country` are the same, even though the `invoice_no` values are different.

In addition, some of these rows have negative values in the `quantity` column. These patterns suggest that these records may represent unusual or system-generated transactions rather than normal customer purchases.

Based on these observations, we have enough evidence to consider removing these rows because they contain missing `customer_id` and `description` values along with other unusual transaction patterns.